Start by downloading the datasets

In [ ]:
# these are the libraries that you will need throughout the assignment
import numpy as np 
import pandas as pd

import matplotlib.pyplot as plt 
import seaborn as sns
%matplotlib inline


In [ ]:
hive_data = pd.DataFrame() # change this
hive_data= pd.read_csv('hiveDownload_ppm_2026_SECOND.csv', header=1, sep=";", engine="python")
#header=1 bc we omit row 0, column names start at row=1

hive_data.shape

In [ ]:
hive_data.head(5)

In [ ]:
PPMI_data = pd.DataFrame()
PPMI_data=pd.read_csv('PPMI_Curated_Data_Cut_Public_20240129 copy.csv', sep=";", engine="python")
PPMI_data.shape


In [ ]:
PPMI_data.head(5)

In [ ]:
field_strength = pd.DataFrame()
field_strength=pd.read_csv('Field_strength_PPMI.csv', sep=";", engine="python")
field_strength.shape

In [ ]:
field_strength.head(5)

In [ ]:
#We only want BL/m00 (baseline) for both PPMI_data and hive_data sets
PPMI_bl= PPMI_data[PPMI_data["EVENT_ID"] == "BL"].copy().reset_index(drop=True)

hive_bl=hive_data[hive_data["TimePoint"] == "m00"].copy().reset_index(drop=True)    
#reset index so numbering is consistent

field_strength_bl=field_strength[field_strength["Timepoint"] == "m00"].copy().reset_index(drop=True)


In [ ]:
hive_bl.shape

In [ ]:
PPMI_bl.shape

In [ ]:
field_strength.shape

In [ ]:
print('Data Show Info\n')
PPMI_bl.info()

In [ ]:
print('Data Show Info\n')
hive_bl.info()

In [ ]:
print('Data Show Info\n')
field_strength_bl.info()

changing all of the supposed numerical values of HIVE dataset to numerical values (from string), except ID, etc. 

In [ ]:
import pandas as pd

def to_float_decimal_comma(s: pd.Series) -> pd.Series:
    return pd.to_numeric(
        s.astype("string").str.strip().str.replace(",", ".", regex=False),
        errors="coerce"
    )

# -------------------
# HIVE
# -------------------
hive_string_cols = [
    "Individual",
    "QCVerdict", "Comment", "Rater", "TimePoint", "APOE",
    "SNCA_rs356181", "SNCA_rs3910105", "MAPT"
]
hive_string_cols = [c for c in hive_string_cols if c in hive_bl.columns]
hive_bl[hive_string_cols] = hive_bl[hive_string_cols].astype("string")

# numeric-candidate cols = everything else
hive_numeric_cols = hive_bl.columns.difference(hive_string_cols)

# find numeric-candidate cols that contain commas anywhere
comma_cols = [
    c for c in hive_numeric_cols
    if hive_bl[c].astype("string").str.contains(",", na=False).any()
]

print("HIVE numeric columns containing commas:", comma_cols)

# convert comma-cols with comma->dot handling
for c in comma_cols:
    hive_bl[c] = to_float_decimal_comma(hive_bl[c])

# convert the rest normally
remaining_cols = hive_numeric_cols.difference(comma_cols)
hive_bl[remaining_cols] = hive_bl[remaining_cols].apply(pd.to_numeric, errors="coerce")


# -------------------
# PPMI
# -------------------
ppmi_string_cols = ["PATNO", "subgroup", "EVENT_ID"]
ppmi_string_cols = [c for c in ppmi_string_cols if c in PPMI_bl.columns]
PPMI_bl[ppmi_string_cols] = PPMI_bl[ppmi_string_cols].astype("string")

# fix decimal comma in these specific PPMI columns
for col in ["age", "age_at_visit", "ageonset", "agediag", "BMI", "duration", "duration_yrs", 
            "age_DATSCAN", "age_LP", "age_upsit"]:
    if col in PPMI_bl.columns:
        PPMI_bl[col] = to_float_decimal_comma(PPMI_bl[col])

print(hive_bl.dtypes)

## -------------------
# field_strength
# -------------------
field_strength_string_cols = ["GUID", "Project", "Individual", "Timepoint", "Scan Type", "Scanner", "Scan date"]
field_strength_string_cols = [c for c in field_strength_string_cols if c in field_strength_bl.columns]
field_strength_bl[field_strength_string_cols] = field_strength_bl[field_strength_string_cols].astype("string")

for col in ["Field Strength", "Rating"]:
    if col in field_strength_bl.columns:
        field_strength_bl[col] = to_float_decimal_comma(field_strength_bl[col])

print("\ndtypes field strength dataset:")
print(field_strength_bl.dtypes)


In [ ]:
#Continuing, we want to make sure that the IDs between the two datasets match: 
#We will compare age, sex, and race columns (the columns that overlap between the datasets)
#And IDs as well ("EVENT_ID" and "TimePoint")
#"age" = age at baseline
#"SEX" = 1=male, 0=female in PPMI, "gen" = 1=male 2=female in HIVE (checked for IDs 3000 and 3001, and they match in the dataset)
#"race" 1=white, 2=black, 3=asian 4=other  .=unknown --> same in both datasets

#Start by comparing IDs
id_col_PPMI = "PATNO"
id_col_hive = "Individual"
id_col_fs = "Individual"

ids_PPMI = set(PPMI_bl[id_col_PPMI].astype(str).str.strip())
ids_hive = set(hive_bl[id_col_hive].astype(str).str.strip())
ids_fs = set(field_strength_bl[id_col_fs].astype(str).str.strip())

#parise comparison
only_in_PPMI = ids_PPMI - ids_hive
only_in_hive = ids_hive - ids_PPMI
in_both = ids_PPMI & ids_hive

#new: include field strength
only_in_fs = ids_fs - ids_hive
fs_and_hive = ids_fs & ids_hive
fs_and_ppmi = ids_fs & ids_PPMI
in_all_three = ids_PPMI & ids_hive & ids_fs

print("IDs only in PPMI:", len(only_in_PPMI))
print("IDs only in HIVE:", len(only_in_hive))
print("IDs only in Field Strength:", len(ids_fs - (ids_PPMI | ids_hive)))

print("IDs in both PPMI and HIVE:", len(in_both))
print("IDs in both FS and HIVE:", len(fs_and_hive))
print("IDs in both FS and PPMI:", len(fs_and_ppmi))

print("IDs in all three datasets:", len(in_all_three))



In [ ]:
#Check for duplicate IDs
print("Number of duplicate IDs in PPMI data: ", PPMI_bl[id_col_PPMI].duplicated().sum())
print("Number of duplicate IDs in HIVE data: ", hive_bl[id_col_hive].duplicated().sum())
print("Number of duplicate IDs in field strength data: ", field_strength_bl[id_col_fs].duplicated().sum())

There are 0 duplicate IDs in the PPMI dataset
The HIVE dataset has 271 duplicates
--> we then want to check which ids are duplicated at baseline and how many times they appear in the dataset

no = number of...

In [ ]:
#no_dups.. = number of duplicates
no_dups_hive_bl = hive_bl[hive_bl["TimePoint"].eq("m00")].copy()

no_dups_rows_hive = no_dups_hive_bl[id_col_hive].duplicated().sum()
no_dups_ids_hive = no_dups_hive_bl.loc[no_dups_hive_bl[id_col_hive].duplicated(keep=False), id_col_hive].nunique()

print("Duplicate baseline rows: ", no_dups_rows_hive)
print("IDs with >1 baseline row: ", no_dups_ids_hive)
#There are 239 unique IDs that appear 2 or more times at m00

#See the top duplicated IDs
no_dups_hive_bl[id_col_hive].value_counts().head(10)

In [ ]:
#make sure age/gen/race/EDUCYRS, educ are consistent within an ID in hive: 
shared = ["age", "gen", "race", "EDUCYRS", "educ"] 
consistency = hive_bl.groupby(id_col_hive)[shared].nunique(dropna=False)
inconsistent = consistency[(consistency > 1).any(axis=1)]
inconsistent.head()  #in hive dataset, list inconsistencies between those variables

The only difference between the variables in the HIVE duplicates are the MRI values (looked at manually) --> now we want to check which duplicates we want to use, by looking at the QCVerdict column

In [ ]:
#Look at "QCVerdict" column in hive dataset. Pass means it could be good to use 

#id_col="Individual" #hive dataset
dup_mask=hive_bl[id_col_hive].duplicated(keep=False)
dups=hive_bl.loc[dup_mask, [id_col_hive, "QCVerdict"]].copy()

pass_counts = dups.assign(is_pass=dups["QCVerdict"].astype(str).str.lower().eq("pass")) \
                 .groupby(id_col_hive)["is_pass"].sum() \
                 .sort_values(ascending=False)
pass_counts.head(20)

### removing fail cases, duplicate etc from HIVE dataset

In [ ]:
#limiting the QCVerdict fails (not just duplicares) and handling the duplicates in hive_bl:
#eliminate if fail QCVerdict, the prioritze pass, drop the IDs with 0 pass, and for the 2+ pass,
#pick the duplicate with the least number of missing MRI values, and then if we need another tiebreaker
#we will just pick the 1st row for that duplicate that satisfies everything else

#1. Keep only rows that PASS (drops Fail + NaN + anything else)
is_pass = (                             #is_pass is a pandas series of True/false with the same length as hive_bl
    hive_bl["QCVerdict"].astype(str).str.strip().str.lower().eq("pass")
)

hive_bl2 = hive_bl.loc[is_pass].copy() #keeps only the rows where "is_pass" is true

#Preserve original order for final tiebreaker 
hive_bl2["_orig_order"] = np.arange(len(hive_bl2))

#2. For handling duplicates (now all remaining rows are Pass after "is_pass"):
#   - if an ID has 1 remaining row --> keep it
#   - if an ID has 2+ remaining rows --> keep the least missing; tie -> earliest original row

#COunt missing values (exclude helper cols; include everything else)
cols_for_missing = [c for c in hive_bl2.columns if c != "_orig_order"]
hive_bl2["_na_count"] = hive_bl2[cols_for_missing].isna().sum(axis=1)

#Sort so "best" row per ID is first: least missing, then earliest orignial row
h_sorted = hive_bl2.sort_values([id_col_hive, "_na_count", "_orig_order"], ascending=[True, True, True])

#keep 1 row per ID
hive_bl3 = (
    h_sorted
    .drop_duplicates(subset=[id_col_hive], keep="first")
    .sort_values("_orig_order")
    .drop(columns=["_orig_order", "_na_count"])
    )

### SANITY CHECKS
print("Original rows:", len(hive_bl))
print("After PASS filter:", len(hive_bl2))
print("Final rows (1 per ID):", len(hive_bl3))
print("Remining duplicate IDs (should be 0):", hive_bl3[id_col_hive].duplicated().sum())


In [ ]:
# Keep baseline only
fs_bl_only = field_strength_bl[field_strength_bl["Timepoint"].eq("m00")].copy()

# Preserve original order for tie-breaking
fs_bl_only["_orig_order"] = np.arange(len(fs_bl_only))

# Count missing values
cols_for_missing_fs = [c for c in fs_bl_only.columns if c != "_orig_order"]
fs_bl_only["_na_count"] = fs_bl_only[cols_for_missing_fs].isna().sum(axis=1)

# Sort so the "best" row per ID comes first
fs_sorted = fs_bl_only.sort_values(
    [id_col_fs, "_na_count", "_orig_order"],
    ascending=[True, True, True]
)

# Keep one row per Individual
field_strength_bl3 = (
    fs_sorted
    .drop_duplicates(subset=[id_col_fs], keep="first")
    .sort_values("_orig_order")
    .drop(columns=["_orig_order", "_na_count"])
)

# Optional: keep only IDs also present in cleaned HIVE
field_strength_bl3_matched = field_strength_bl3[
    field_strength_bl3[id_col_fs].astype(str).str.strip().isin(
        hive_bl3[id_col_hive].astype(str).str.strip()
    )
].copy()

# Checks
print("Original field strength rows:", len(field_strength_bl))
print("Baseline field strength rows:", len(fs_bl_only))
print("Final field strength rows (1 per ID):", len(field_strength_bl3))
print("Remaining duplicate IDs:", field_strength_bl3[id_col_fs].duplicated().sum())
print("Rows matched to hive_bl3 IDs:", len(field_strength_bl3_matched))

### Next STEP
- Remove all of the IDs from PPMI dataset that don't match the IDs existing in the HIVE dataset. 
- Lastly check if the IDs match between the two datasets

In [ ]:
# create comparable ID columns
PPMI_bl["id_col_"] = PPMI_bl[id_col_PPMI].astype("string").str.strip()
hive_bl3["id_col_"] = hive_bl3[id_col_hive].astype("string").str.strip()
field_strength_bl3["id_col_"] = field_strength_bl3["Individual"].astype("string").str.strip()

# drop missing IDs properly
ppmi_ids = set(PPMI_bl["id_col_"].dropna())
hive_ids = set(hive_bl3["id_col_"].dropna())
fs_ids = set(field_strength_bl3["id_col_"].dropna())

# first check HIVE vs Field Strength
shared_hive_fs = hive_ids & fs_ids
only_in_hive = hive_ids - fs_ids
only_in_fs = fs_ids - hive_ids

print("Shared IDs between HIVE and Field Strength:", len(shared_hive_fs))
print("IDs only in HIVE:", len(only_in_hive))
print("IDs only in Field Strength:", len(only_in_fs))

# keep only matching IDs in HIVE and Field Strength
hive_keep = hive_bl3.loc[hive_bl3["id_col_"].isin(shared_hive_fs)].copy()
field_strength_keep = field_strength_bl3.loc[field_strength_bl3["id_col_"].isin(shared_hive_fs)].copy()

# now find IDs shared across all three datasets
shared_ids_all = ppmi_ids & hive_ids & fs_ids

PPMI_keep = PPMI_bl.loc[PPMI_bl["id_col_"].isin(shared_ids_all)].copy()
hive_keep_all = hive_bl3.loc[hive_bl3["id_col_"].isin(shared_ids_all)].copy()
field_strength_keep_all = field_strength_bl3.loc[field_strength_bl3["id_col_"].isin(shared_ids_all)].copy()

print("Shared IDs across PPMI, HIVE, and Field Strength:", len(shared_ids_all))
print("PPMI rows kept:", len(PPMI_keep), "unique IDs:", PPMI_keep["id_col_"].nunique())
print("HIVE rows kept:", len(hive_keep_all), "unique IDs:", hive_keep_all["id_col_"].nunique())
print("Field Strength rows kept:", len(field_strength_keep_all), "unique IDs:", field_strength_keep_all["id_col_"].nunique())

In [ ]:
print(PPMI_keep["id_col_"].nunique())
print(hive_keep["id_col_"].nunique())
print(field_strength_keep["id_col_"].nunique())

In [ ]:
PPMI_keep.shape

In [ ]:
hive_keep.shape

In [ ]:
field_strength_keep.shape

In [ ]:
#SANITY CHECK 
print(hive_keep["gen"].dtype)
print(hive_keep["gen"].value_counts(dropna=False).head(20))
print(hive_keep["gen"].unique()[:20])

In [ ]:
import pandas as pd
import numpy as np

# -------------------------
# Harmonize demographics on the filtered subsets
# -------------------------

# HIVE gen: 1=male, 2=female -> 1=male, 0=female (to match PPMI SEX)
g = pd.to_numeric(hive_keep["gen"], errors="coerce")  # keeps 1.0/2.0/NaN
g = g.replace({2.0: 0, 2: 0})  # female -> 0 (handles float or int)
hive_keep["gen"] = g.where(g.isin([0, 1])).astype("Int64")

print(hive_keep["gen"].value_counts(dropna=False))

# HIVE race: ensure integer codes 1-4 (optional but keeps types consistent)
if "race" in hive_keep.columns:
    hive_keep["race"] = (
        hive_keep["race"]
        .astype("string").str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
    )
    hive_keep["race"] = pd.to_numeric(hive_keep["race"], errors="coerce").astype("Int64")

# PPMI SEX: ensure Int64 0/1 (usually already, but enforce)
if "SEX" in PPMI_keep.columns:
    PPMI_keep["SEX"] = (
        PPMI_keep["SEX"]
        .astype("string").str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
    )
    PPMI_keep["SEX"] = pd.to_numeric(PPMI_keep["SEX"], errors="coerce").astype("Int64")
    PPMI_keep["SEX"] = PPMI_keep["SEX"].where(PPMI_keep["SEX"].isin([0, 1])).astype("Int64")

# PPMI age/race: enforce numeric (optional)
if "age" in PPMI_keep.columns:
    PPMI_keep["age"] = pd.to_numeric(PPMI_keep["age"], errors="coerce")

if "race" in PPMI_keep.columns:
    PPMI_keep["race"] = (
        PPMI_keep["race"]
        .astype("string").str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .replace({"": pd.NA, "nan": pd.NA, "None": pd.NA})
    )
    PPMI_keep["race"] = pd.to_numeric(PPMI_keep["race"], errors="coerce").astype("Int64")

In [ ]:
missing_count_ppmi = PPMI_keep[['age', 'SEX', 'race']].isna().sum()
missing_count_hive = hive_keep[['age', 'gen', 'race']].isna().sum()
missing_count_fs = field_strength_keep[["Field Strength"]].isna().sum()

print("Missing in PPMI:\n", missing_count_ppmi)
print("Missing in HIVE:\n", missing_count_hive)
print("missing in FS:\n", missing_count_fs)


In [ ]:
cmp = (
    PPMI_keep[["id_col_", "age", "SEX", "race"]]
    .merge(
        hive_keep[["id_col_", "age", "gen", "race"]],
        on="id_col_",
        how="inner",
        suffixes=("_ppmi", "_hive")
    )
)

# ---- mismatch flags (ignore missing on either side) ----
sex_mismatch = (
    cmp["SEX"].notna() & cmp["gen"].notna() &
    (cmp["SEX"] != cmp["gen"])
)

race_mismatch = (
    cmp["race_ppmi"].notna() & cmp["race_hive"].notna() &
    (cmp["race_ppmi"] != cmp["race_hive"])
)

tol = 1.0  # years
age_mismatch = (
    cmp["age_ppmi"].notna() & cmp["age_hive"].notna() &
    ((cmp["age_ppmi"] - cmp["age_hive"]).abs() > tol)
)

diff = cmp[sex_mismatch | race_mismatch | age_mismatch].copy()
diff["age_diff"] = (diff["age_ppmi"] - diff["age_hive"]).abs()

print("Rows with any mismatch:", len(diff))
diff[["id_col_", "SEX", "gen", "race_ppmi", "race_hive", "age_ppmi", "age_hive", "age_diff"]].head(20)

Now that the IDs are aligned and the demographic columns (age, sex, race) are consistent, the next step is:
- Merge HIVE and PPMI 

In [ ]:
#We can now merge the 2 datasets, but only matching ID 
#don't include age, race and gen/sex columns from hive. 

# 1) one row per ID in each (first wins; change keep=... if you prefer)
PPMI_1 = PPMI_keep.drop_duplicates("id_col_", keep="first").copy()

HIVE_1 = (
    hive_keep
    .drop(columns=[c for c in ["age", "race", "gen", id_col_hive] if c in hive_keep.columns])  # drop hive age/race + raw hive ID col
    .drop_duplicates("id_col_", keep="first")
    .copy() 
)  

FIELD_STRENGTH_1 = (
    field_strength_keep
    .drop(columns=[c for c in [id_col_fs] if c in field_strength_keep.columns])
    .drop_duplicates("id_col_", keep="first")
    .copy()
)

# 2) merge on shared key
merged = (
    PPMI_1
    .merge(HIVE_1, on="id_col_", how="inner", suffixes=("", "_hive"))
    .merge(FIELD_STRENGTH_1, on="id_col_", how="inner", suffixes=("", "_fs"))
)

# 3) final cleanup:
# keep PATNO as the ID; optionally drop id_col_ if you don’t want it
# merged = merged.drop(columns=["id_col_"])

print("Merged rows:", len(merged))
print("Unique PATNO:", merged["PATNO"].nunique())
print("Any duplicate PATNO rows?:", merged["PATNO"].duplicated().sum())
print("Columns:", merged.columns.tolist())

In [ ]:
print("Final shape:", merged.shape)
print("Unique IDs:", merged["id_col_"].nunique())

Next step: 
- Freeze the final dataset containing:
    - 1635 unique IDs that's matching from both PPMI and HIVE data
    - 537 columns (3 less than if age, gen and race from both datasets where to be saved), but since matching/checking if they overlap, we assume that they are the same for each of the 1635 IDs
- We don't tuch merged_full again
- no duplicates, all at baseline, 

In [ ]:
merged.head(10)

In [ ]:
final_df = merged.copy()

In [ ]:
out_path = "final_df.csv"

final_df.to_csv(
    out_path,
    index=False,        # don’t write pandas index as a column
    encoding="utf-8",   # safe default
    na_rep=""           # empty cells instead of "NaN"
)

print(f"Saved {len(final_df)} rows to {out_path}")